## Week 7 NYTimes API & Webscraping part 1
Megan October 12,2026

In [1]:
import requests # this gives us the ability to request data from remote websites
from time import sleep # this gives us the ability to implement time delays in our code
from getpass import getpass #this is a password-protection feature for python notebooks

key = getpass("type in your key here")

type in your key here ········


In [2]:
query = '"Artifical Intelligence"' # This is the search term we are looking for in any of the metadata

begin_date = "19000901" # this is the begindate for our search (September 1 2026)
end_date = "20260930" # This is the enddate for our search (September 30th 2026)
sort = 'oldest'
url = f"https://api.nytimes.com/svc/search/v2/articlesearch.json?q={query}&begin_date={begin_date}&end_date={end_date}&api-key={key}&sort={sort}"
response = requests.get(url)
print(response.status_code)

200


In [7]:
parsed_nytimes_response = response.json() # interpret responses as python-readable JSON
print(parsed_nytimes_response.keys()) # tells me what keys are in the raw response
print(parsed_nytimes_response['response'].keys()) # tells me what keys are in the "response" parent
nytimes_docs = parsed_nytimes_response['response']['docs'] #tells me what keys ate in the "docs" key, which is a child of "response"
print(len(nytimes_docs))

dict_keys(['status', 'copyright', 'response'])
dict_keys(['docs', 'metadata'])
10


In [9]:
print(nytimes_docs[1]['headline'])
print(nytimes_docs[1]['headline']['main'])
print(nytimes_docs[1]['pub_date'])
print(nytimes_docs[1].keys())

{'main': 'Fran Bell Is Bride of Richard Waldinger', 'kicker': '', 'print_headline': 'Fran Bell Is Bride of Richard Waldinger'}
Fran Bell Is Bride of Richard Waldinger
1975-01-19T05:00:00Z
dict_keys(['abstract', 'byline', 'document_type', 'headline', '_id', 'keywords', 'multimedia', 'news_desk', 'print_page', 'print_section', 'pub_date', 'section_name', 'snippet', 'source', 'subsection_name', 'type_of_material', 'uri', 'web_url', 'word_count'])


## Webscraping with beautiful soup
below are the libraries i nee to import before i begin web scraping

In [ ]:



import csv
import json
import os # operating system library, let's us do things like making new folders
import requests
from time import sleep
from bs4 import BeautifulSoup # the scraping library!



Below is some example HTML that we can traverse using the beauitifulsoup function

In [13]:

import csv
import json
import os # operating system library, let's us do things like making new folders
import requests
from time import sleep
from bs4 import BeautifulSoup # the scraping library!


html = """
<html>
  <body>
    <h1>Adeline Ravoux</h1>
    <p class="artist">Vincent van Gogh</p>
    <p class="date">1890</p>
    <a href="/art/1958.31">Artwork page</a>
  </body>
</html>
"""

soup = BeautifulSoup(html, "html.parser")  # html.parser is the built-in parser that comes with python



In [16]:
title = soup.find("h1")
print(title)
print(title.get_text())

<h1>Adeline Ravoux</h1>
Adeline Ravoux


In [18]:
paragraph_elements = soup.find_all("p")
print(paragraph_elements)

[<p class="artist">Vincent van Gogh</p>, <p class="date">1890</p>]


In [22]:
artist = soup.find("p",class_="artist") #underscore after class, becuase it's a reserved term in python
print(artist.get_text())
date = artist.find_next_sibling() 
print(date.get_text())

Vincent van Gogh
1890


# understanding robots permissions
we can view the permissions that a website grants to do things like automaticallyt access data with a bot - robots.txt file

In [ ]:
headers = {"User-Agent": "INFO664-class-project (educational)"}   # you can call this whatever you want, it shows up on the scraped website's logs as who you are (ie not a Google/AI bot or web browser)

os.makedirs("scraped_pages", exist_ok=True)  # make a new folder to hold our scraped website HTML data. exist_ok = True means, if the directory already exists, do nothing and don't return an error

def fetch_page(url, filename):  # declaring a new function, which takes a URL to be scraped, and a destination filename, as arguments
    if os.path.exists(filename):  # check to see if file already exists, if so, don't download it from the web, load it instead
        with open(filename, "r", encoding="utf-8") as file:
            return file.read()  # in this case, return provides the read file data, and escapes from the function, ignoring everything below

    response = requests.get(url, headers=headers) # grab the file, using our pre-defined headers as the header data

    if not response.ok: # if the response reports an error
        print(url, "failed with status code", response.status_code) # tell us why it failed
        return None # return a "None" object, since there's nothing usable in the response, and escape from this function
    with open(filename, "w",  encoding="utf-8") as file:
        file.write(response.text)

    sleep(10) # wait 10 seconds, since our robots.txt file asks for a 10 second Crawl-delay
    return response.text

